# IndicOCR Full-Precision (FP) OCR on Kaggle GPU

This notebook is **100% self-contained**: it automatically downloads the model weights from Hugging Face (`bodhan-ai/indic-ocr`), fetches benchmark images from `ai4bharat/indicdlp` (or uses uploaded images), extracts text crops with Stage 1 layout detection, and runs full-precision (FP16/BF16) OCR on the GPU in **~100–200ms per crop**.

### Instructions
1. In the Kaggle top menu, go to **Add-ons** -> **Secrets** and add your secret:
   - **Label**: `HUGGING_FACE_API_KEY`
   - **Value**: `<your Hugging Face access token>`
2. In the right sidebar under **Notebook settings**:
   - **Accelerator**: Select **GPU T4 x2** (or **P100 / A100**)
   - **Internet**: Toggle **On**
3. Click **Run All** (or run cells step-by-step).
4. Download `ocr_ref_cache_<N>crops.json` from the output directory and drop it into your local `fixtures/baseline_cache/` folder.

## 1. Install & Upgrade Dependencies (Requires Transformers >= 5.16)

In [ ]:
!pip uninstall -y torchaudio
!pip install -U -q "transformers>=5.16.0" accelerate safetensors huggingface-hub pyarrow tqdm


## 2. Verify Transformers Version & Reload Kernel if Needed

In [ ]:
import os
import transformers

print(f"Transformers version: {transformers.__version__}")

try:
    from transformers.models.pp_doclayout_v3.modeling_pp_doclayout_v3 import PPDocLayoutV3ForObjectDetection
    from transformers import PPDocLayoutV3Config, PPDocLayoutV3ForObjectDetection
    print("PPDocLayoutV3 components loaded successfully!")
except Exception as err:
    import traceback
    traceback.print_exc()
    print("Note: If you just installed packages, restart via Session -> Restart Session")


## 3. Authenticate with Hugging Face via Kaggle Secrets

In [ ]:
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGING_FACE_API_KEY")
    if hf_token:
        login(token=hf_token)
        os.environ["HF_TOKEN"] = hf_token
        print("Successfully authenticated with Hugging Face via Kaggle Secrets!")
    else:
        print("Warning: HUGGING_FACE_API_KEY is empty.")
except Exception as err:
    print(f"Could not load secret from Kaggle Secrets: {err}")
    print("Make sure 'HUGGING_FACE_API_KEY' is added in Kaggle menu: Add-ons -> Secrets.")

## 4. Check GPU Acceleration

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
else:
    dtype = "float32"
print(f"Precision dtype: {dtype}")

## 5. Download Model Weights & Helper Modules from Hugging Face

In [ ]:
import sys
from pathlib import Path
from huggingface_hub import snapshot_download

BUNDLE_DIR = Path("./model_bundle").resolve()
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading bodhan-ai/indic-ocr from Hugging Face...")
snapshot_download(
    repo_id="bodhan-ai/indic-ocr",
    local_dir=str(BUNDLE_DIR),
    allow_patterns=[
        "config.json",
        "*.py",
        "weights/layout/*",
        "weights/ocr/*",
        "schemas/*",
        "assets/*",
    ],
    local_dir_use_symlinks=False,
)

if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
print("Model bundle downloaded successfully and loaded into path.")

## 6. Benchmark Images (Auto-download or Use Uploaded)

In [ ]:
import io
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download
import pyarrow.parquet as pq

IMAGES_DIR = Path("./fixtures/benchmark_images").resolve()
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

NUM_IMAGES = 25  # Number of document images to process

# Check if user already uploaded images to /kaggle/input
uploaded_candidates = list(Path("/kaggle/input").glob("**/*.png")) if Path("/kaggle/input").exists() else []

if uploaded_candidates:
    image_paths = sorted(uploaded_candidates)[:NUM_IMAGES]
    print(f"Found {len(image_paths)} uploaded images in /kaggle/input")
else:
    print("Downloading benchmark images from ai4bharat/indicdlp on Hugging Face...")
    DATASET_REPO = "ai4bharat/indicdlp"
    api = HfApi()
    all_files = api.list_repo_files(DATASET_REPO, repo_type="dataset")
    parquet_shards = sorted([f for f in all_files if f.startswith("data/test-") and f.endswith(".parquet")])

    image_paths = []
    for shard in parquet_shards:
        if len(image_paths) >= NUM_IMAGES:
            break
        local_shard = hf_hub_download(repo_id=DATASET_REPO, filename=shard, repo_type="dataset")
        table = pq.read_table(local_shard, columns=["image"])
        for row_idx in range(len(table)):
            if len(image_paths) >= NUM_IMAGES:
                break
            img_bytes = table["image"][row_idx].as_py()["bytes"]
            out_name = f"indicdlp_bench_{len(image_paths):04d}.png"
            out_path = IMAGES_DIR / out_name
            with Image.open(io.BytesIO(img_bytes)) as img:
                img.save(out_path, format="PNG")
            image_paths.append(out_path)

    print(f"Downloaded {len(image_paths)} document images into {IMAGES_DIR}")

## 7. Extract Document Crops with Stage 1 Layout Detector

In [ ]:
from idp_offline import IndicDocLayout
from idp_layout import IndicDocLayoutBackend
from idp_recognizer import build_requests
from idp_contract import is_transcribed
from idp_types import LayoutConfig, CropConfig

TARGET_CROPS = 50  # Number of text crops to transcribe (e.g. 5, 25, 50, 100)

layout_weights = BUNDLE_DIR / "weights" / "layout"
layout_cfg = LayoutConfig(device="cpu")
layout_backend = IndicDocLayoutBackend(str(layout_weights), config=layout_cfg)
layout_engine = IndicDocLayout(backend=layout_backend)
crop_cfg = CropConfig()

crop_requests = []
crop_metadata = []

for img_path in sorted(image_paths):
    if len(crop_requests) >= TARGET_CROPS:
        break
    res = layout_engine.detect(str(img_path))
    img = Image.open(img_path).convert("RGB")
    eligible = [b for b in res.blocks if is_transcribed(b.label)]
    if not eligible:
        continue
    reqs, orders = build_requests(eligible, img, crop_cfg, table_format="markdown")
    for req, order in zip(reqs, orders):
        if len(crop_requests) >= TARGET_CROPS:
            break
        crop_requests.append(req)
        crop_metadata.append({
            "image": img_path.name,
            "order": order,
            "prompt": req.prompt,
        })

layout_engine.close()
print(f"Prepared {len(crop_requests)} document crop requests.")

## 8. Initialize Full-Precision OCR Model on GPU

In [ ]:
from idp_recognizer import HfRecognizer
from idp_types import RecognizerConfig

ocr_weights = BUNDLE_DIR / "weights" / "ocr"
cfg = RecognizerConfig(dtype=dtype, max_tokens=256)

print(f"Loading HfRecognizer on {device} with {dtype} precision...")
recognizer = HfRecognizer(
    ckpt=str(ocr_weights),
    config=cfg,
    device=device,
    attn_implementation="sdpa" if device == "cuda" else "eager",
)
print("Recognizer ready on GPU!")

## 9. Fast GPU Transcription with Live Progress

In [ ]:
import time
from tqdm.auto import tqdm

results = []
latencies_ms = []

t_start = time.perf_counter()
for idx, req in enumerate(tqdm(crop_requests, desc="GPU Transcription")):
    t0 = time.perf_counter()
    transcription = recognizer.transcribe([req])[0]
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    results.append(transcription)
    latencies_ms.append(elapsed_ms)

total_elapsed_sec = time.perf_counter() - t_start
mean_ms = sum(latencies_ms) / len(latencies_ms) if latencies_ms else 0.0
print(f"\nTranscribed {len(results)} crops in {total_elapsed_sec:.2f}s (Average: {mean_ms:.1f} ms per crop)!")

## 10. Save Baseline Cache Files

The output file `ocr_ref_cache_<N>crops.json` can be downloaded directly from Kaggle and placed in your local `fixtures/baseline_cache/` folder.

In [ ]:
import json

out_dir = Path("./output").resolve()
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Format matching fixtures/baseline_cache/ocr_ref_cache_<N>crops.json
cache_filename = f"ocr_ref_cache_{len(results)}crops.json"
cache_path = out_dir / cache_filename
with open(cache_path, "w", encoding="utf-8") as f:
    json.dump({"texts": results, "pt_ms": sum(latencies_ms)}, f, indent=2, ensure_ascii=False)

# Also save in the current working directory for easy Kaggle download
with open(cache_filename, "w", encoding="utf-8") as f:
    json.dump({"texts": results, "pt_ms": sum(latencies_ms)}, f, indent=2, ensure_ascii=False)

# 2. Detailed report with metadata
detailed_report = {
    "device": device,
    "dtype": dtype,
    "total_crops": len(results),
    "total_time_seconds": round(total_elapsed_sec, 2),
    "mean_latency_ms": round(mean_ms, 2),
    "crops": [
        {
            "index": i,
            "image": meta["image"],
            "order": meta["order"],
            "latency_ms": round(lat, 2),
            "char_count": len(txt),
            "text": txt,
        }
        for i, (meta, txt, lat) in enumerate(zip(crop_metadata, results, latencies_ms))
    ],
}
detailed_path = out_dir / "kaggle_fp_ocr_report.json"
with open(detailed_path, "w", encoding="utf-8") as f:
    json.dump(detailed_report, f, indent=2, ensure_ascii=False)

print(f"Cache file saved to: {cache_path}")
print(f"Detailed report saved to: {detailed_path}")
print(f"\nDownload '{cache_filename}' and place it in your local 'fixtures/baseline_cache/{cache_filename}'.")

## 11. Sample Transcription Previews

In [ ]:
for i in range(min(5, len(results))):
    print(f"--- Crop {i+1} ({crop_metadata[i]['image']}) [{latencies_ms[i]:.1f} ms] ---")
    print(results[i])
    print()